In [3]:
import numpy as np
import torch

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [6]:
import torch
import torchvision
import torchvision.transforms as transforms

In [54]:
import torch.nn as nn
import torch.nn.functional as F

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=0)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # compute flattened size:
        # input: 200x200
        # after conv3x3 (no padding): 198x198
        # after 2x2 maxpool: 99x99
        self.flattened_size = 32 * 99 * 99

        self.fc1 = nn.Linear(self.flattened_size, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # (N, 32, 99, 99)
        x = torch.flatten(x, 1)                # (N, 32*99*99)
        x = F.relu(self.fc1(x))                # (N, 64)
        x = self.fc2(x)       # (N, 1) for binary classification
        return x

model = Net()

In [55]:
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.002, momentum=0.8)

In [56]:
# Option 1: Using torchsummary (install with: pip install torchsummary)
from torchsummary import summary
summary(model, input_size=(3, 200, 200))

# Option 2: Manual counting
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params}")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
         MaxPool2d-2           [-1, 32, 99, 99]               0
            Linear-3                   [-1, 64]      20,072,512
            Linear-4                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 11.96
Params size (MB): 76.57
Estimated Total Size (MB): 89.00
----------------------------------------------------------------
Total parameters: 20073473


In [57]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ) # ImageNet normalization
])

train_dataset = datasets.ImageFolder(root="./data/data/train/", transform=train_transforms)
validation_dataset   = datasets.ImageFolder(root="./data/data/test/", transform=train_transforms)

print("Classes:", train_ds.classes)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
validation_loader   = DataLoader(validation_dataset, batch_size=32, shuffle=False)

Classes: ['curly', 'straight']


In [58]:
device="cpu"

In [59]:
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6202, Acc: 0.6438, Val Loss: 0.6253, Val Acc: 0.6368
Epoch 2/10, Loss: 0.5549, Acc: 0.7225, Val Loss: 0.6161, Val Acc: 0.6418
Epoch 3/10, Loss: 0.4952, Acc: 0.7525, Val Loss: 0.6855, Val Acc: 0.6468
Epoch 4/10, Loss: 0.4289, Acc: 0.8187, Val Loss: 0.5904, Val Acc: 0.6965
Epoch 5/10, Loss: 0.3792, Acc: 0.8550, Val Loss: 0.7130, Val Acc: 0.6169
Epoch 6/10, Loss: 0.3741, Acc: 0.8462, Val Loss: 0.9281, Val Acc: 0.5970
Epoch 7/10, Loss: 0.4068, Acc: 0.8187, Val Loss: 0.5880, Val Acc: 0.7164
Epoch 8/10, Loss: 0.2996, Acc: 0.8725, Val Loss: 0.6804, Val Acc: 0.7164
Epoch 9/10, Loss: 0.2091, Acc: 0.9275, Val Loss: 0.6849, Val Acc: 0.7015
Epoch 10/10, Loss: 0.1866, Acc: 0.9337, Val Loss: 0.7270, Val Acc: 0.7363


In [62]:
import numpy as np
print(np.median(history['acc']))

0.8325


In [63]:
print(np.std(history['loss']))

0.13243352287328858


In [64]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),transforms.RandomRotation(50),
transforms.RandomResizedCrop(200, scale=(0.9, 1.0), ratio=(0.9, 1.1)),
transforms.RandomHorizontalFlip() # ImageNet normalization
])

train_dataset = datasets.ImageFolder(root="./data/data/train/", transform=train_transforms)
validation_dataset   = datasets.ImageFolder(root="./data/data/test/", transform=train_transforms)

print("Classes:", train_ds.classes)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
validation_loader   = DataLoader(validation_dataset, batch_size=32, shuffle=False)

Classes: ['curly', 'straight']


In [65]:
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1) # Ensure labels are float and have shape (batch_size, 1)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        # For binary classification with BCEWithLogitsLoss, apply sigmoid to outputs before thresholding for accuracy
        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6454, Acc: 0.6825, Val Loss: 0.6129, Val Acc: 0.7164
Epoch 2/10, Loss: 0.5956, Acc: 0.6587, Val Loss: 0.5587, Val Acc: 0.7114
Epoch 3/10, Loss: 0.5581, Acc: 0.7000, Val Loss: 0.5971, Val Acc: 0.6617
Epoch 4/10, Loss: 0.5307, Acc: 0.7188, Val Loss: 0.5922, Val Acc: 0.6965
Epoch 5/10, Loss: 0.5323, Acc: 0.7262, Val Loss: 0.5878, Val Acc: 0.7164
Epoch 6/10, Loss: 0.5120, Acc: 0.7312, Val Loss: 0.5834, Val Acc: 0.6866
Epoch 7/10, Loss: 0.5038, Acc: 0.7350, Val Loss: 0.5713, Val Acc: 0.7114
Epoch 8/10, Loss: 0.5184, Acc: 0.7188, Val Loss: 0.5889, Val Acc: 0.6816
Epoch 9/10, Loss: 0.5145, Acc: 0.7375, Val Loss: 0.5744, Val Acc: 0.7214
Epoch 10/10, Loss: 0.4955, Acc: 0.7462, Val Loss: 0.6273, Val Acc: 0.6667


In [66]:
print(np.mean(history['val_loss']))

0.5893855126194694


In [67]:
print(np.mean(history['val_acc'][5:10]))

0.6935323383084577
